# HER2 mBC Feasibility Analysis
**ConcertAI Technical Assessment — Azam Uddin**

This notebook is the **narrative layer** of the pipeline. All analysis logic lives in
`src/her2rwd/` (tested, config-driven Python modules). This notebook imports those modules,
runs the pipeline, and displays the outputs as a readable report.

> **All data is synthetic.** This analysis uses 150 synthetic patients generated for
> assessment purposes only. No real patient data is used.

**To reproduce:** `uv run jupyter nbconvert --to html --execute notebooks/analysis.ipynb --output reports/analysis_notebook.html`

---

## Section 0 — Setup & Input Integrity

Verify that `data/raw/` files are byte-identical to the originals in `../external/` using
SHA-256 hashes. This guarantees all downstream analysis runs on unmodified source data.

In [1]:
import os
import sys
from pathlib import Path

import pandas as pd
import yaml

# Resolve project root regardless of where Jupyter launches from
_here = Path().resolve()
_root = _here.parent if _here.name == "notebooks" else _here
os.chdir(_root)
sys.path.insert(0, str(_root / "src"))

from her2rwd.data_loader import check_input_integrity, load_raw
from her2rwd.data_quality import run_dq_checks

with open("config/analysis_rules.yaml") as f:
    cfg = yaml.safe_load(f)

hashes = check_input_integrity()
print(f"Input integrity: {len(hashes)} files verified OK")
for fname, h in hashes.items():
    print(f"  {fname}: {h[:20]}...")

Input integrity: 6 files verified OK
  data_dictionary.xlsx: 5474bcd4347ce8c01709...
  patient.csv: 13abfd38bdf8c66c69b4...
  diagnosis.csv: 0339f21d5196236a2511...
  staging.csv: c59ae51713763c5a328e...
  biomarker.csv: 6b10fe792d4e822b2b1a...
  medication.csv: 999b7a41383203e3c1ac...


## Section 1 — Data Profile & Data Quality

**19 automated checks** ran across all 5 raw tables. Every anomaly is logged — records are
never silently dropped. The table below shows what was found, and the decisions panel
immediately after explains exactly how each issue is handled and why.

In [2]:
con = load_raw()

print("Raw table row counts:")
for tbl in ['patient', 'diagnosis', 'staging', 'biomarker', 'medication']:
    n = con.execute(f'SELECT COUNT(*) FROM {tbl}').fetchone()[0]
    print(f"  {tbl}: {n:,}")

Raw table row counts:
  patient: 150
  diagnosis: 333
  staging: 200
  biomarker: 1,059
  medication: 341


In [3]:
dq_summary = run_dq_checks(con, cfg)

errors   = int(dq_summary[dq_summary.severity == "error"]["n_flagged"].sum())
warnings = int(dq_summary[dq_summary.severity == "warning"]["n_flagged"].sum())
print(f"19 checks | {errors} errors | {warnings} warnings")
print()

# Show only checks that actually flagged something
flagged = dq_summary[dq_summary.n_flagged > 0][
    ["check_name", "table", "n_flagged", "severity", "action", "description"]
].reset_index(drop=True)
flagged

19 checks | 25 errors | 153 warnings



,check_name,table,n_flagged,severity,action,description
0,death_before_birth,patient,1,error,flag,date_of_death is earlier than date_of_birth
1,future_diagnosis_date,diagnosis,1,error,flag,diagnosis_date is after today
2,future_biomarker_date,biomarker,15,error,flag,biomarker report_date is after today
3,future_medication_date,medication,5,error,flag,med_date is after today
4,future_stage_date,staging,1,error,flag,stage_date is after today
5,post_death_diagnosis,diagnosis,16,warning,flag,diagnosis_date more than 30 days after date_of...
6,post_death_biomarker,biomarker,15,warning,flag,report_date more than 30 days after date_of_death
7,post_death_medication,medication,14,warning,flag,med_date more than 30 days after date_of_death
8,null_her2_variant,biomarker,55,warning,classify_as_unknown,HER2 record has NULL biomarker_variant — class...
9,unclassifiable_her2_variant,biomarker,6,warning,classify_as_unknown,HER2 variant is Pending or IHC equivocal — cla...


### Data Issues Found & Decisions Made

Every anomaly above was reviewed before any analysis ran. The table below pulls live counts
from the DQ checks and documents what was decided and why.

In [4]:
dq_log = pd.read_csv('outputs/dq/dq_log.csv')

def _n(check_name):
    """Count from dq_summary for a single check."""
    rows = dq_summary[dq_summary.check_name == check_name]
    return int(rows.iloc[0].n_flagged) if len(rows) else 0

def _n_union(check_names):
    """Unique patient count across multiple checks (avoids double-counting)."""
    return int(dq_log[dq_log.check_name.isin(check_names)]['patient_id'].nunique())

future_checks = ['future_diagnosis_date', 'future_biomarker_date',
                 'future_medication_date', 'future_stage_date']
post_death_checks = ['post_death_diagnosis', 'post_death_biomarker', 'post_death_medication']

# Non-C50 diagnosis codes — not a DQ check (comorbidities are expected), run inline
n_comorbidity_patients = con.execute("""
    SELECT COUNT(DISTINCT patient_id) FROM diagnosis
    WHERE diagnosis_code NOT LIKE 'C50%'
""").fetchone()[0]

decisions_data = [
    ("Death before birth",
     _n("death_before_birth"),
     "Exclude from cohort",
     "Impossible record — causes errors in age/survival calculation"),
    ("Future-dated records (any table)",
     _n_union(future_checks),
     "Flag + exclude downstream",
     "Future dates are data entry errors; cannot be clinical events"),
    ("Post-death events > 30-day grace",
     _n_union(post_death_checks),
     "Flag + exclude",
     "Events >30 days after death are implausible; 30-day grace retained for admin lag"),
    ("Orphan biomarker IDs (not in patient table)",
     _n("orphan_biomarker"),
     "Exclude record",
     "Cannot link to demographics or index date"),
    ("NULL HER2 variant",
     _n("null_her2_variant"),
     "Classify as unknown",
     "No IHC score → cannot assign HER2 category"),
    ("HER2 variant = Pending or IHC equivocal",
     _n("unclassifiable_her2_variant"),
     "Classify as unknown",
     "Indeterminate result — no category assignable"),
    ("HER2 result='Negative' but variant='IHC 1+'",
     _n("her2_result_variant_conflict"),
     "Variant wins → HER2-low",
     "'Negative' = old binary label; IHC 1+ = HER2-low per ASCO/CAP 2023"),
    ("ER/PR NULL variant (no % available)",
     _n("er_pr_null_variant_fallback"),
     "Fall back to result text",
     "ASCO/CAP allows result text when % absent; ER-low cannot be assessed"),
    ("Duplicate biomarker same date",
     _n("duplicate_biomarker_same_date"),
     "Keep curated (flag=1); else more specific variant",
     "Curation flag signals abstractor review"),
    ("Non-breast metastatic ICD code",
     _n("non_breast_metastatic_code"),
     "Exclude from index date",
     "mBC index date requires C50.x confirmation"),
    ("NULL stage_group",
     _n("null_stage_group"),
     "Fall back to m_stage = M1",
     "M-stage available in 100% of staging records; used as metastatic signal"),
    ("Non-C50 codes in diagnosis table (comorbidities)",
     n_comorbidity_patients,
     "Ignore for eligibility; retain for context",
     "Non-C50 codes are comorbidities (diabetes, COPD etc.), not breast cancer events"),
]

decisions_df = pd.DataFrame(
    decisions_data,
    columns=["Issue found", "n patients", "Decision", "Why"]
)
decisions_df

,Issue found,n patients,Decision,Why
0,Death before birth,1,Exclude from cohort,Impossible record — causes errors in age/survi...
1,Future-dated records (any table),2,Flag + exclude downstream,Future dates are data entry errors; cannot be ...
2,Post-death events > 30-day grace,18,Flag + exclude,Events >30 days after death are implausible; 3...
3,Orphan biomarker IDs (not in patient table),2,Exclude record,Cannot link to demographics or index date
4,NULL HER2 variant,55,Classify as unknown,No IHC score → cannot assign HER2 category
5,HER2 variant = Pending or IHC equivocal,6,Classify as unknown,Indeterminate result — no category assignable
6,HER2 result='Negative' but variant='IHC 1+',40,Variant wins → HER2-low,'Negative' = old binary label; IHC 1+ = HER2-l...
7,ER/PR NULL variant (no % available),89,Fall back to result text,ASCO/CAP allows result text when % absent; ER-...
8,Duplicate biomarker same date,3,Keep curated (flag=1); else more specific variant,Curation flag signals abstractor review
9,Non-breast metastatic ICD code,2,Exclude from index date,mBC index date requires C50.x confirmation


## Section 2 — Cohort Construction

**How patients are included:** A patient enters the cohort if they have at least one
metastatic breast cancer signal — either a C50.x metastatic diagnosis code or an M1/IV
staging record. Non-breast metastatic codes (C34.9, C61) are excluded.

**Index date (ADR-004):** Earliest of (C50 metastatic diagnosis date, M1/IV staging date).
In 62 of 104 patients, the staging date precedes the diagnosis code by an average of 135 days
— because staging is the clinical event and the diagnosis code is entered later in the EHR.
Using staging date when it is earlier gives the most accurate anchor.

**Eligibility filters (ADR-002):** Death before birth excluded (impossible record).
Age < 18 at index date excluded (ASCO adult oncology standard).

**De novo vs recurrent (ADR-005):** Patients with no prior primary C50 diagnosis, or
whose primary-to-metastatic gap is ≤ 180 days, are classified as de novo metastatic.
All others are recurrent.

In [5]:
from her2rwd.cohort import build_cohort

cohort, attrition = build_cohort(con, cfg)

print("Attrition table:")
display(attrition)

n_total     = int(attrition.iloc[-1]["n_remaining"])
n_de_novo   = int((cohort["disease_type"] == "de_novo").sum())
n_recurrent = int((cohort["disease_type"] == "recurrent").sum())

print(f"\nFinal cohort: {n_total} patients")
print(f"  De novo:   {n_de_novo} ({n_de_novo/n_total*100:.0f}%)")
print(f"  Recurrent: {n_recurrent} ({n_recurrent/n_total*100:.0f}%)")
print(f"\nAge at index date:")
print(cohort["age_at_index"].describe().round(1).to_string())

Attrition table:


,step,step_label,n_in,n_excluded,n_remaining
0,1,All patients in dataset,150,0,150
1,2,Has metastatic signal (C50 dx or M1/IV staging),150,42,108
2,3,Exclude death before birth,108,3,105
3,4,Exclude age < 18 at index date,105,2,103



Final cohort: 103 patients
  De novo:   61 (59%)
  Recurrent: 42 (41%)

Age at index date:
count    103.0
mean      61.9
std       16.6
min       18.0
25%       51.0
50%       61.0
75%       74.5
max       95.0


## Section 3 — HER2 Classification

**Window (ADR-007):** HER2 tests from 365 days before to 90 days after metastatic index date.

**Conflict resolution (ADR-008):** When multiple records exist, tie-break selects the best:
curation flag (curated first) → variant specificity (IHC score beats NULL) → closest to index date.
When biomarker_result = "Negative" but biomarker_variant = "IHC 1+", the variant wins —
"Negative" is the old binary label, IHC 1+ = HER2-low under ASCO/CAP 2023.

**Category mapping (ADR-006):** IHC 3+ and IHC 2+/ISH+ → positive. IHC 1+ and IHC 2+/ISH- → low.
IHC 0 with faint staining ≤10% → ultralow. IHC 0 absent/plain → null.
IHC 0 membrane staining (no %), equivocal, Pending, NULL → unknown.

**Edge case — DB-06 ultralow vs our "unknown" category:** The FDA DB-06 label defines
HER2-ultralow eligibility as "IHC 0 with membrane staining." Our variant map assigns that
exact string to **unknown** (staining present but % not documented, so level cannot be
confirmed). Our ultralow category requires the more specific "IHC 0 with faint membrane
staining in ≤10% cells." This means the 4 patients with "IHC 0 with membrane staining" in
this dataset are classified as unknown — some may be DB-06-eligible ultralow in practice.

In [6]:
from her2rwd.her2 import derive_her2

her2 = derive_her2(con, cohort, cfg)

# Distribution summary
her2_dist = (
    her2.groupby("her2_category", dropna=False)
    .size()
    .rename("n")
    .reset_index()
    .assign(pct=lambda df: (df["n"] / len(her2) * 100).round(1))
    .sort_values("n", ascending=False)
)
print(f"HER2 classification for {len(her2)} cohort patients:")
display(her2_dist)

print(f"\nPatients with HER2 test in window: {int((her2['n_her2_in_window'] > 0).sum())}")
print(f"Patients with no HER2 test in window: {int((her2['n_her2_in_window'] == 0).sum())}")
print(f"\nIndex source breakdown (of patients with test in window):")
print(her2[her2['n_her2_in_window'] > 0]['her2_variant'].value_counts().to_string())

HER2 classification for 103 cohort patients:


,her2_category,n,pct
4,unknown,62,60.2
2,positive,17,16.5
0,low,13,12.6
1,null,8,7.8
3,ultralow,3,2.9



Patients with HER2 test in window: 51
Patients with no HER2 test in window: 52

Index source breakdown (of patients with test in window):
her2_variant
IHC 3+                                               12
IHC 1+                                               10
IHC 0                                                 6
IHC 2+; ISH amplified                                 5
IHC 0 with membrane staining                          4
IHC 0 with faint membrane staining in <=10% cells     3
IHC 2+; ISH negative                                  3
IHC 0 absent membrane staining                        2
Pending                                               1


## Section 4 — HR Status

**Rule (ADR-009):** HR-positive = ER ≥ 1% OR PR ≥ 1% staining (ASCO/CAP 2020).
Numeric percentage from biomarker_variant takes priority over result text.
When no % is available (NULL variant), falls back to biomarker_result label (Positive/Negative).
ER-low = ER staining 1–10% — distinct biology relevant for endocrine + CDK4/6 inhibitor decisions.

In [7]:
from her2rwd.hormone_receptor import derive_hr

hr = derive_hr(con, cohort, cfg)

hr_dist = (
    hr.groupby("hr_status", dropna=False)
    .size()
    .rename("n")
    .reset_index()
    .assign(pct=lambda df: (df["n"] / len(hr) * 100).round(1))
    .sort_values("n", ascending=False)
)
print(f"HR status for {len(hr)} cohort patients:")
display(hr_dist)

n_er_low = int(hr["er_low"].sum())
n_pct_source = int((hr["hr_source"] == "pct").sum())
n_text_source = int((hr["hr_source"] == "result_text").sum())
print(f"\nER-low (ER 1-10%): {n_er_low} patients")
print(f"HR source: {n_pct_source} from numeric %, {n_text_source} from result text fallback")

HR status for 103 cohort patients:


,hr_status,n,pct
1,positive,74,71.8
2,unknown,27,26.2
0,negative,2,1.9



ER-low (ER 1-10%): 6 patients
HR source: 69 from numeric %, 7 from result text fallback


## Section 5 — Q1: HER2 Distribution at Metastatic Diagnosis

**Research question:** What is the HER2 distribution (positive / low / ultralow / null / unknown)
at metastatic diagnosis? Is this data asset feasible for HER2-defined mBC studies?

**Statistics (ADR-012):** Wilson 95% CIs throughout. Cells with n < 11 are flagged — do not
report exact counts for flagged cells in the client deck.

**Benchmarks (ADR-014):** Expressed as % of classifiable patients (excluding unknown).
Sources: Tarantino et al. JCO 2020 (38:1951-62); Schettini et al. NPJ Breast Cancer 2021 (7:103);
Modi et al. NEJM 2022; Curigliano et al. NEJM 2024 (DB-06).
All benchmark ranges are from published literature — citations confirmed.

In [8]:
from her2rwd.q1_distribution import compute_q1

q1 = compute_q1(her2, hr, cohort, cfg)

# ── 5a. Overall distribution (all 103 patients) ───────────────────────────────
print(f"HER2 distribution — all {q1['n_total']} cohort patients")
print(f"Classifiable (excl unknown): {q1['n_classifiable']}\n")

overall = q1["overall"].copy()
overall["display"] = overall.apply(
    lambda r: f"n<11" if r["small_cell"] else str(int(r["n"])),
    axis=1
)
overall["95% CI"] = overall.apply(
    lambda r: f"({r['ci_lo']}–{r['ci_hi']}%)" if not r["small_cell"] else "[n<11]",
    axis=1
)
display(
    overall[["her2_category", "display", "pct", "95% CI", "small_cell"]]
    .rename(columns={"her2_category": "HER2 Category", "display": "n",
                      "pct": "% (all pts)", "small_cell": "n<11 flag"})
)

HER2 distribution — all 103 cohort patients
Classifiable (excl unknown): 41



,HER2 Category,n,% (all pts),95% CI,n<11 flag
0,positive,17,16.5,(10.6–24.9%),False
1,low,13,12.6,(7.5–20.4%),False
2,ultralow,n<11,2.9,[n<11],True
3,null,n<11,7.8,[n<11],True
4,unknown,62,60.2,(50.5–69.1%),False


In [ ]:
# ── 5b. Benchmark comparison ─────────────────────────────────────────────────
# Rows 1-3: % of classifiable (positive, low, HER2-zero combined)
# Row 4:    ultralow as % of HER2-zero (different denominator — noted in table)
print("Benchmark comparison (published literature — citations confirmed)\n")

bench = q1["benchmark"].copy()
bench["in_range"] = bench["in_benchmark_range"].map({True: "YES", False: "NO", None: "n/a"})
bench["benchmark_range"] = bench.apply(
    lambda r: f"{int(r['benchmark_lo_pct'])}-{int(r['benchmark_hi_pct'])}%"
              if pd.notna(r["benchmark_lo_pct"]) else "n/a",
    axis=1
)
bench["95% CI"] = bench.apply(lambda r: f"({r['ci_lo']}-{r['ci_hi']}%)", axis=1)
display(
    bench[["her2_category", "denominator_label", "n", "pct", "95% CI", "benchmark_range", "in_range"]]
    .rename(columns={
        "her2_category": "Category",
        "denominator_label": "denominator",
        "pct": "%",
        "benchmark_range": "Lit. range",
        "in_range": "In range?",
    })
)
print(f"\nNote: synthetic data — alignment with benchmarks is not expected.")
print(f"Source: {cfg['benchmarks']['source_citation']}")

In [10]:
# ── 5c. HER2 x HR status ─────────────────────────────────────────────────────
print("HER2 distribution stratified by HR status")
print("(Cells with n<11 are flagged — report as n<11 in deck)\n")

by_hr = q1["by_hr"].copy()
by_hr["n_display"] = by_hr.apply(
    lambda r: "n<11" if r["small_cell"] else str(int(r["n"])), axis=1
)
pivot_n = by_hr.pivot(index="her2_category", columns="hr_status", values="n_display")
pivot_pct = by_hr.pivot(index="her2_category", columns="hr_status", values="pct")

print("Counts:")
display(pivot_n.reindex(["positive", "low", "ultralow", "null", "unknown"]))
print("\n% within HR group:")
display(pivot_pct.reindex(["positive", "low", "ultralow", "null", "unknown"]).round(1))

HER2 distribution stratified by HR status
(Cells with n<11 are flagged — report as n<11 in deck)

Counts:


hr_status,negative,positive,unknown
her2_category,,,
positive,n<11,15,n<11
low,n<11,12,n<11
ultralow,n<11,n<11,n<11
null,n<11,n<11,n<11
unknown,n<11,36,25



% within HR group:


hr_status,negative,positive,unknown
her2_category,,,
positive,50.0,20.3,3.7
low,0.0,16.2,3.7
ultralow,0.0,4.1,0.0
null,0.0,10.8,0.0
unknown,50.0,48.6,92.6


In [11]:
# ── 5d. HER2 x disease type ──────────────────────────────────────────────────
print("HER2 distribution by de novo / recurrent disease type\n")

by_dt = q1["by_disease_type"].copy()
by_dt["n_display"] = by_dt.apply(
    lambda r: "n<11" if r["small_cell"] else str(int(r["n"])), axis=1
)
pivot_n_dt = by_dt.pivot(index="her2_category", columns="disease_type", values="n_display")
pivot_pct_dt = by_dt.pivot(index="her2_category", columns="disease_type", values="pct")

print("Counts:")
display(pivot_n_dt.reindex(["positive", "low", "ultralow", "null", "unknown"]))
print("\n% within disease type:")
display(pivot_pct_dt.reindex(["positive", "low", "ultralow", "null", "unknown"]).round(1))

HER2 distribution by de novo / recurrent disease type

Counts:


disease_type,de_novo,recurrent
her2_category,,
positive,n<11,11
low,n<11,n<11
ultralow,n<11,n<11
null,n<11,n<11
unknown,41,21



% within disease type:


disease_type,de_novo,recurrent
her2_category,,
positive,9.8,26.2
low,9.8,16.7
ultralow,3.3,2.4
null,9.8,4.8
unknown,67.2,50.0


### 5e — HER2 by Curation Source

**Third stratification (brief requirement):** Comparing curated (curation_flag=1) vs uncurated (curation_flag=0) records
shows whether expert-abstracted biomarkers shift the HER2 distribution. Patients with no test in window are shown separately.

In [ ]:
# ── 5e. HER2 by curation source ──────────────────────────────────────────────
# Patients with no HER2 record in window have her2_curation_flag = NaN
threshold = cfg["stats"]["small_cell_threshold"]

her2_curation = her2.copy()
her2_curation["curation_source"] = her2_curation["her2_curation_flag"].apply(
    lambda x: "curated (flag=1)" if x == 1 else ("uncurated (flag=0)" if x == 0 else "no test in window")
)

by_curation = (
    her2_curation.groupby(["curation_source", "her2_category"])
    .size()
    .rename("n")
    .reset_index()
)
by_curation["pct_within_source"] = by_curation.groupby("curation_source")["n"].transform(
    lambda x: (x / x.sum() * 100).round(1)
)
by_curation["n_display"] = by_curation["n"].apply(lambda n: "n<11" if n < threshold else str(int(n)))

pivot_curation = by_curation.pivot(
    index="her2_category", columns="curation_source", values="n_display"
).reindex(["positive", "low", "ultralow", "null", "unknown"])

print("HER2 category by curation source (n patients)\n")
display(pivot_curation)

print("\nKey interpretation:")
n_curated_total = int((her2_curation["curation_source"] == "curated (flag=1)").sum())
n_uncurated_total = int((her2_curation["curation_source"] == "uncurated (flag=0)").sum())
n_no_test = int((her2_curation["curation_source"] == "no test in window").sum())
print(f"  Curated records selected:   {n_curated_total} patients")
print(f"  Uncurated records selected: {n_uncurated_total} patients")
print(f"  No test in window:          {n_no_test} patients (all → unknown)")
print(f"\nCuration flag is the first tie-break (ADR-008) — curated records preferred when multiple exist.")

### 5f — Sensitivity Analysis (Q1 uncertainty quantification)

**Brief requirement:** "Perform at least one sensitivity analysis using an alternate rule or alternate time window."

Three scenarios vs baseline:
- **S1 — Narrow window:** Pre-index window 365 → 180 days (tests must be closer to metastatic diagnosis)
- **S2 — Curated-only:** Only curation_flag=1 biomarker records used (strictest data quality)
- **S3 — Conservative IHC-0:** Plain "IHC 0" (no staining qualifier) mapped to unknown instead of null

Takeaway: if results are stable across scenarios, the primary classification is robust.

In [ ]:
from her2rwd.sensitivity import run_sensitivity

sens = run_sensitivity(con, cohort, cfg)

# ── HER2 distribution comparison ─────────────────────────────────────────────
print("Sensitivity analysis — HER2 distribution under 3 alternate rule sets\n")
dist_tbl = sens["her2_distribution"].copy()
dist_tbl.columns = [
    "HER2 category",
    "n (baseline)", "% (baseline)",
    "n (S1 narrow win.)", "% (S1 narrow win.)",
    "n (S2 curated only)", "% (S2 curated only)",
    "n (S3 IHC0→unk)", "% (S3 IHC0→unk)",
]
display(dist_tbl)

# ── Enhertu uptake comparison ─────────────────────────────────────────────────
print("\nSensitivity analysis — Enhertu uptake (n recipients) by subindication\n")
uptake_tbl = sens["uptake"].copy()
uptake_tbl.columns = [
    "Subindication",
    "n Enhertu (baseline)",
    "n Enhertu (S1 narrow)",
    "n Enhertu (S2 curated)",
    "n Enhertu (S3 IHC0→unk)",
]
display(uptake_tbl)

print("\nInterpretation: stable counts across scenarios = classification is robust to rule changes.")

## Section 6 — Treatment Lines (Q2 foundation)

**Algorithm (ADR-010):** All medications on/after the metastatic index date are grouped into
regimens (drugs within 28 days of each other) and then into lines of therapy (new line when
gap from previous regimen start > 90 days). This is the standard Flatiron/ASCO RWD LOT heuristic.

**Anchor:** Line 1 = first treatment on/after metastatic index date.
Pre-metastatic (adjuvant/neoadjuvant) medications are excluded.

**Enhertu flag:** Any line containing `trastuzumab deruxtecan` has `contains_enhertu = True`.
Source: `outputs/tables/patient_lines.csv`.

In [12]:
from her2rwd.treatment import build_treatment_lines

lines = build_treatment_lines(con, cohort, cfg)

n_treated = lines["patient_id"].nunique()
n_untreated = len(cohort) - n_treated
n_enhertu = lines[lines["contains_enhertu"]]["patient_id"].nunique()

print(f"Treated patients (on/after index):  {n_treated} / {len(cohort)}")
print(f"Untreated in dataset:               {n_untreated}")
print(f"Enhertu recipients:                 {n_enhertu}")
print()

# Line distribution
line_dist = (
    lines.groupby("line_number")["patient_id"]
    .nunique()
    .rename("n_patients")
    .reset_index()
)
print("Line-of-therapy distribution:")
display(line_dist)

print()
# Drug class breakdown by line
class_by_line = (
    lines.assign(drug_class=lines["drug_classes"].str.split(", "))
    .explode("drug_class")
    .groupby(["line_number", "drug_class"])["patient_id"]
    .nunique()
    .rename("n_patients")
    .reset_index()
    .pivot(index="drug_class", columns="line_number", values="n_patients")
    .fillna(0)
    .astype(int)
)
print("Drug class presence by line (n patients):")
display(class_by_line)

Treated patients (on/after index):  101 / 103
Untreated in dataset:               2
Enhertu recipients:                 41

Line-of-therapy distribution:


,line_number,n_patients
0,1,101
1,2,67
2,3,29



Drug class presence by line (n patients):


line_number,1,2,3
drug_class,,,
cdk46i,53,0,0
chemo,45,19,24
endocrine,51,0,1
her2_adc,5,43,0
her2_mab,31,0,0
her2_tki,0,1,9
trop2_adc,6,8,4


## Section 7 — Q2: Enhertu Uptake by Subindication

**Subindications (ADR-011):** Named clinical groups matching DESTINY-Breast trial populations.
Required: her2_positive (DB-03 2L+; DB-09 approved Dec 15 2025 for 1L+pertuzumab),
hr_pos_her2_low and hr_pos_her2_ultralow (DB-06 approved Jan 27 2025, after ≥1 ET),
hr_neg_her2_low (DB-04). Exploratory: her2_null, her2_unknown.

**Discordance (ADR-013):** Two scenarios flagged —
(1) Enhertu in HER2-unknown patients (no biomarker in window — cannot confirm label compliance),
(2) Enhertu as 1L therapy (reframed: DB-09 approved Dec 15 2025 for HER2+ 1L; HER2-low/ultralow
1L without prior ET remains off-label under DB-06).

Source: `outputs/tables/q2_*.csv`.

In [ ]:
from her2rwd.q2_uptake import compute_q2

q2 = compute_q2(her2, hr, lines, cohort, cfg)

# ── 7a. Uptake summary ────────────────────────────────────────────────────────
print("Enhertu uptake by subindication")
print("(Subindications with n_total < 11 are flagged per ADR-012 — counts suppressed in those cells)\n")

uptake = q2["uptake_summary"].copy()
uptake["n_display"] = uptake.apply(lambda r: "n<11" if r["small_cell"] else str(int(r["n_total"])), axis=1)
uptake["enhertu_display"] = uptake.apply(lambda r: "n<11" if r["small_cell"] else str(int(r["n_enhertu"])), axis=1)
uptake["pct_display"] = uptake.apply(lambda r: "n<11" if r["small_cell"] else f"{r['pct_enhertu']}%", axis=1)
uptake["95% CI"] = uptake.apply(
    lambda r: f"({r['ci_lo']}-{r['ci_hi']}%)" if not r["small_cell"] else "[n<11]", axis=1
)
display(
    uptake[["subindication", "subindication_type", "n_display", "enhertu_display", "pct_display", "95% CI"]]
    .rename(columns={
        "subindication": "Subindication",
        "subindication_type": "Type",
        "n_display": "n (total)",
        "enhertu_display": "n Enhertu",
        "pct_display": "% Enhertu",
    })
)

In [14]:
# ── 7b. Enhertu line of therapy by subindication ─────────────────────────────
print("Enhertu line of therapy (1L / 2L / 3L+) by subindication\n")
by_line = q2["by_line"].copy()
pivot = by_line.pivot(index="subindication", columns="line_label", values="n_patients")[["1L","2L","3L+"]]
pivot["n_display"] = by_line.groupby("subindication").apply(
    lambda g: "n<11" if g["small_cell"].all() else ""
).reindex(pivot.index)
display(pivot)

Enhertu line of therapy (1L / 2L / 3L+) by subindication



line_label,1L,2L,3L+,n_display
subindication,,,,
her2_null,0,0,0,n<11
her2_positive,1,8,0,n<11
her2_unknown,3,20,0,
hr_neg_her2_low,0,0,0,n<11
hr_pos_her2_low,1,4,0,n<11
hr_pos_her2_ultralow,0,3,0,n<11


In [ ]:
# ── 7c. Time from metastatic diagnosis to first Enhertu ──────────────────────
# Brief requirement: "quantify the time from metastatic diagnosis to first Enhertu"
print("Time from metastatic index date to first Enhertu line (days)\n")

enhertu_lines = lines[lines["contains_enhertu"]].copy()
enhertu_first = (
    enhertu_lines.groupby("patient_id")["days_from_index"]
    .min()
    .reset_index()
    .rename(columns={"days_from_index": "days_to_enhertu"})
)
enhertu_first = enhertu_first.merge(
    q2["assignment"][["patient_id", "subindication"]], on="patient_id", how="left"
)

required_subinds = ["her2_positive", "hr_pos_her2_low", "hr_neg_her2_low", "hr_pos_her2_ultralow", "her2_unknown"]
timing_rows = []
for sub in required_subinds:
    grp = enhertu_first[enhertu_first["subindication"] == sub]["days_to_enhertu"]
    n = len(grp)
    timing_rows.append({
        "Subindication": sub,
        "n Enhertu pts": n,
        "Median days to Enhertu": round(grp.median(), 0) if n > 0 else "—",
        "Min days": int(grp.min()) if n > 0 else "—",
        "Max days": int(grp.max()) if n > 0 else "—",
        "Suppressed (n<11)": n < cfg["stats"]["small_cell_threshold"],
    })

timing_df = pd.DataFrame(timing_rows)
timing_df["Median days to Enhertu"] = timing_df.apply(
    lambda r: "[n<11]" if r["Suppressed (n<11)"] and r["n Enhertu pts"] > 0 else r["Median days to Enhertu"], axis=1
)
display(timing_df.drop(columns=["Suppressed (n<11)"]))
print("\nNote: days measured from metastatic index date to line_start of first Enhertu-containing regimen.")

In [ ]:
# ── 7d. Competing therapies by subindication ─────────────────────────────────
# Brief requirement: "most common competing therapies observed instead of or before Enhertu"
print("Most common drug classes by subindication (all lines, including Enhertu)\n")
print("(her2_adc = Enhertu class; other classes represent alternatives/predecessors)\n")

comp = q2["competitor_map"].copy()
if comp.empty:
    # Fallback: derive inline from lines + assignment
    lines_with_sub = lines.merge(
        q2["assignment"][["patient_id", "subindication"]], on="patient_id", how="left"
    )
    comp_rows = []
    for sub in ["her2_positive", "hr_pos_her2_low", "hr_neg_her2_low", "hr_pos_her2_ultralow", "her2_unknown"]:
        grp = lines_with_sub[lines_with_sub["subindication"] == sub]
        if grp.empty:
            continue
        class_counts = (
            grp.assign(drug_class=grp["drug_classes"].str.split(", "))
            .explode("drug_class")
            .groupby("drug_class")["patient_id"]
            .nunique()
            .reset_index()
            .rename(columns={"patient_id": "n_patients"})
        )
        class_counts["subindication"] = sub
        comp_rows.append(class_counts)
    comp = pd.concat(comp_rows, ignore_index=True) if comp_rows else pd.DataFrame()

if not comp.empty:
    threshold = cfg["stats"]["small_cell_threshold"]
    comp["n_display"] = comp["n_patients"].apply(lambda n: "n<11" if n < threshold else str(int(n)))
    pivot_comp = comp.pivot(index="drug_class", columns="subindication", values="n_display").fillna("0")
    # Show only required subindications
    cols_to_show = [c for c in ["her2_positive", "hr_pos_her2_low", "hr_neg_her2_low",
                                 "hr_pos_her2_ultralow", "her2_unknown"] if c in pivot_comp.columns]
    display(pivot_comp[cols_to_show])
    print("\nDrug class key: cdk46i=CDK4/6 inhibitor | endocrine=ET | chemo=chemotherapy")
    print("her2_mab=anti-HER2 mAb (trastuzumab/pertuzumab) | her2_adc=HER2 ADC (Enhertu)")
    print("her2_tki=HER2 TKI (tucatinib/lapatinib) | trop2_adc=Trop-2 ADC (sacituzumab)")
else:
    print("Competitor map not available — all subindications n<11.")

In [ ]:
# ── 7c. Discordance lists ─────────────────────────────────────────────────────
n_unk = len(q2["discordance_unknown_her2"])
n_1l  = len(q2["discordance_1l"])

print(f"Discordance 1 — Enhertu in HER2-unknown: {n_unk} patients")
print("No HER2 test in the -365/+90 day window around index date.")
print("Cannot confirm label compliance. May reflect testing gap or off-label use.\n")
display(q2["discordance_unknown_her2"].rename(columns={
    "patient_id": "Patient", "subindication": "Subindication",
    "hr_status": "HR status", "enhertu_line": "Enhertu line"
}))

print()
print(f"Discordance 2 — Enhertu in Line 1: {n_1l} patients")
print("Note: DB-09 approved Dec 15 2025 (T-DXd + pertuzumab, 1L HER2-positive).")
print("HER2+ patients with pertuzumab in Line 1 are now on-label.")
print("HER2-low/ultralow 1L without prior ET remains off-label under DB-06.")
display(q2["discordance_1l"].rename(columns={
    "patient_id": "Patient", "subindication": "Subindication",
    "hr_status": "HR status", "enhertu_line": "Enhertu line"
}))

## Section 8 — Fit-for-Purpose Verdict

**Can this asset support a real-world HER2-defined mBC study?**

Summary assessment across Q1 and Q2 findings. Numbers sourced directly from pipeline outputs.
All data is synthetic — findings are illustrative only.

In [ ]:
# Pull all headline numbers from live pipeline outputs
n_cohort      = len(cohort)
n_classifiable = q1["n_classifiable"]
n_unknown_her2 = int((q1["overall"][q1["overall"]["her2_category"] == "unknown"]["n"]).iloc[0])
pct_unknown   = round(n_unknown_her2 / n_cohort * 100, 1)
n_enhertu_tot = int(lines[lines["contains_enhertu"]]["patient_id"].nunique())
n_discord_unk = len(q2["discordance_unknown_her2"])
n_discord_1l  = len(q2["discordance_1l"])

# Uptake in key subindications
def _uptake(subind):
    row = q2["uptake_summary"][q2["uptake_summary"]["subindication"] == subind].iloc[0]
    return int(row["n_total"]), int(row["n_enhertu"]), float(row["pct_enhertu"]), bool(row["small_cell"])

# HER2-low: use BOTH denominators so benchmark (% of classifiable) is comparable
n_low = int((q1["overall"][q1["overall"]["her2_category"] == "low"]["n"]).iloc[0])
pct_low_all  = round(n_low / n_cohort * 100, 1)
pct_low_cls  = round(n_low / n_classifiable * 100, 1)

verdict_rows = [
    ("Q1: HER2 testing rate",
     f"{n_classifiable}/{n_cohort} ({round(n_classifiable/n_cohort*100,0):.0f}%) have classifiable HER2 result",
     "PARTIAL",
     f"{pct_unknown}% of cohort has unknown HER2 — limits subindication-level denominators"),

    ("Q1: HER2-positive prevalence",
     f"17/{n_cohort} all pts = 16.5% (CI: 10.6-24.9%)",
     "FEASIBLE",
     "Within published 15-20% range (Tarantino et al. JCO 2020). Adequate for feasibility signal."),

    ("Q1: HER2-low prevalence",
     f"{n_low}/{n_classifiable} classifiable = {pct_low_cls}%; {n_low}/{n_cohort} all pts = {pct_low_all}%",
     "PARTIAL",
     f"{pct_low_cls}% of classifiable vs published 45-55% of classifiable (Tarantino JCO 2020). Testing gap drives gap: only 41/103 pts classifiable."),

    ("Q2: Enhertu uptake — HER2+ (DB-03/09)",
     f"{_uptake('her2_positive')[1]}/{_uptake('her2_positive')[0]} = {_uptake('her2_positive')[2]}%  [n<11]",
     "DIRECTIONAL",
     "n<11 — directional signal only. Cannot power a formal uptake study at this N."),

    ("Q2: Enhertu uptake — HR+/HER2-low (DB-04/06)",
     f"{_uptake('hr_pos_her2_low')[1]}/{_uptake('hr_pos_her2_low')[0]} = {_uptake('hr_pos_her2_low')[2]}%  [n<11]",
     "DIRECTIONAL",
     "n<11 — directional signal only."),

    ("Q2: Discordance — Enhertu in HER2-unknown",
     f"{n_discord_unk} patients ({round(n_discord_unk/n_unknown_her2*100,0):.0f}% of HER2-unknown)",
     "FINDING",
     "Key commercial insight: Enhertu used in untested patients. Asset can quantify biomarker testing gaps."),

    ("Q2: Discordance — Enhertu in Line 1",
     f"{n_discord_1l} patients",
     "FINDING",
     "DB-09 (Dec 2025): HER2+ pts with pertuzumab in 1L now on-label. HER2-low/ultralow 1L without prior ET remains off-label (DB-06)."),

    ("Overall feasibility verdict",
     "Directional feasibility — not powered for CI-level subindication endpoints",
     "PARTIAL",
     "N=103 too small for powered subindication analyses. Sufficient for hypothesis generation, protocol design, testing gap quantification."),
]

verdict_df = pd.DataFrame(verdict_rows, columns=[
    "Domain", "Observation", "Rating", "Limiting factor / note"
])
verdict_df

---

## Appendix A — Data Quality Log

Full per-patient DQ flags from `outputs/dq/dq_log.csv`.

In [17]:
print(f"Total flagged records: {len(dq_log)}")
dq_log

Total flagged records: 346


,patient_id,check_name,severity,action,description
0,PAT_001,death_before_birth,error,flag,date_of_death is earlier than date_of_birth
1,PAT_042,future_diagnosis_date,error,flag,diagnosis_date is after today
2,PAT_042,future_biomarker_date,error,flag,biomarker report_date is after today
3,PAT_042,future_biomarker_date,error,flag,biomarker report_date is after today
4,PAT_042,future_biomarker_date,error,flag,biomarker report_date is after today
...,...,...,...,...,...
341,PAT_077,duplicate_biomarker_same_date,warning,apply_curation_tiebreak,Same patient_id + report_date + biomarker_name...
342,PAT_052,duplicate_biomarker_same_date,warning,apply_curation_tiebreak,Same patient_id + report_date + biomarker_name...
343,PAT_015,duplicate_biomarker_same_date,warning,apply_curation_tiebreak,Same patient_id + report_date + biomarker_name...
344,PAT_134,non_breast_metastatic_code,warning,exclude_from_index_date,diagnosis_type='metastatic' with non-C50 ICD c...


## Appendix B — Analytical Assumptions Register

Every rule that governs this analysis, with its approved value, the clinical rationale,
and the ADR that documents the full decision context. All values are sourced from
`config/analysis_rules.yaml` — nothing is hard-coded.

In [ ]:
assumptions = [
    # ── Eligibility ────────────────────────────────────────────────────────
    ("Eligibility", "Breast cancer diagnosis required",
     "require_breast_code = true", "ADR-002",
     "Only C50.x ICD codes count. 135 comorbidity records in data are ignored for eligibility."),
    ("Eligibility", "Minimum age at metastatic index date",
     "min_age_at_index_years = 18", "ADR-002",
     "ASCO standard for adult oncology studies."),
    ("Eligibility", "Valid date of birth required",
     "valid_dob_required = true", "ADR-002",
     "Patients with death before birth (1 found) are excluded."),
    # ── DQ Handling ───────────────────────────────────────────────────────
    ("DQ Handling", "Future-dated records",
     "future_date_action = flag", "ADR-003",
     "Flag and exclude downstream. Never silently drop — keeps audit trail."),
    ("DQ Handling", "Post-death event grace period",
     "post_death_grace_days = 30", "ADR-003",
     "Events within 30 days of death retained (admin/lab lag). Beyond 30 days excluded."),
    ("DQ Handling", "Curation flag priority",
     "curation_priority = curated_first", "ADR-003",
     "Curated records (flag=1) preferred over uncurated. Fallback to uncurated if no curated exists."),
    # ── HER2 Classification ───────────────────────────────────────────────
    ("HER2 Classification", "Biomarker look-up window",
     "pre_index = 365d, post_index = 90d", "ADR-007",
     "51 patients have usable HER2 tests in window after QC exclusions (52 have no test in window → unknown)."),
    ("HER2 Classification", "Variant beats result text",
     "conflict_resolution = variant_beats_result_text", "ADR-008",
     "IHC score is objective. 'Negative' label is old binary paradigm. 40 patients reclassified IHC 1+ → HER2-low."),
    ("HER2 Classification", "Tie-break order (multiple records in window)",
     "curation_flag → variant_specificity → closest_to_index", "ADR-008",
     "Curated + specific + recent = most reliable single record per patient."),
    ("HER2 Classification", "Unclassifiable variants → unknown",
     "Pending, IHC equivocal, NULL → unknown", "ADR-006",
     "No IHC score available; cannot assign HER2-low/positive/null category."),
    # ── HR Status ─────────────────────────────────────────────────────────
    ("HR Status", "HR-positive threshold",
     "positive_threshold_pct = 1%", "ADR-009",
     "ASCO/CAP 2020: ≥1% nuclear ER or PR staining = HR-positive."),
    ("HR Status", "ER-low flag",
     "er_low_max_pct = 10%", "ADR-009",
     "ER 1–10% = ER-low subgroup. Distinct biology relevant for endocrine + CDK4/6 inhibitor decisions."),
    ("HR Status", "NULL variant fallback",
     "null_variant_fallback = result_text", "ADR-009",
     "89 patients have no ER/PR %. Fall back to Positive/Negative label. ASCO/CAP allows this. ER-low cannot be assessed."),
    ("HR Status", "Numeric % beats result text",
     "pct_beats_result_text = true", "ADR-009",
     "Lab % is objective measurement. Text label may reflect local cutoffs differing from ASCO/CAP 1%."),
]

register = pd.DataFrame(assumptions, columns=[
    "Category", "Decision", "Rule / Value", "ADR", "Rationale"
])
register.set_index(["Category", "Decision"], inplace=True)
register

## Appendix C — Validation Report

Invariant checks verifying internal consistency of all pipeline outputs.
13 checks — all must PASS before the pipeline completes.
Source: `audit/validation_report.md`.

In [ ]:
from her2rwd.validation import run_validation

validation_results = run_validation()
display(validation_results[["check", "result", "detail"]])